# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
# QUAN TRỌNG: các thay đổi pretrain (gen_all, vision unfreeze, whole-word mask) nằm ở
# NHÁNH exp/pretrain-gen-all — KHÔNG phải main. Không checkout đúng nhánh sẽ bị lỗi
# 'unrecognized arguments: --vision_unfreeze_last_n --mlm_mask_mode'.
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart, rồi chạy tiếp TỪ cell cấu hình.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'                       # <== ĐIỀN token HF của bạn
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall'     # repo sẽ lưu MODEL PRETRAIN (gen_all)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE / MOCK — in debug ĐẦY ĐỦ để đảm bảo các phần chạy đúng
> ⚠️ **Vision unfreeze TẮT** (`--vision_unfreeze_last_n 0`). Có **guard NaN** (pretrain-only) trong forward.
> ✅ **Giảm nạng**: MLM chỉ dùng **câu hỏi** (`mlm_ocr_in_text=False`).
> 🔧 **Đóng góp chính — VG-OCR-Denoise** (`gen_task=denoise`): đưa **OCR bị nhiễu** (char/word) vào nhánh OCR nhưng **giữ nguyên đặc trưng thị giác (det/rec/box)**; decoder phải **sinh lại bản SẠCH** → học **sửa lỗi OCR bằng thị giác** (tên riêng/viết tắt/dấu). `read` = A/B (OCR sạch → đọc).

Bật `TWC_TRAIN_LOG=1`. Trong log tìm:
1. `>>> [pretrain] ... gen_task = denoise (VG-OCR-Denoise (sửa lỗi))`
2. `🔬 [VERIFY]` toàn ✅, `✅ [GEN] gen_loss finite & > 0`, KHÔNG còn `🚨 ... has NaN`
3. Per-step `[Pretrain] ... Loss(M) Loss(I) Loss(TWC) Loss(GEN)` **giảm dần**
4. Cuối (sau train): `🔎 [MLM DEBUG]` (câu hỏi mask → pred) và `🔧 [GEN DEBUG]` (**target sạch vs output** — thấy model sửa lỗi OCR chưa)

In [ ]:
import os, importlib
os.environ['TWC_TRAIN_LOG'] = '1'   # in debug per-step (Loss M/I/TWC/GEN) trong vòng lặp mock
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm: vision unfreeze gây grad-nổ + NaN forward → ổn định trước
    '--mlm_mask_mode', 'wholeword',    # mask trọn từ (bỏ 'nạng' copy subword)
    '--gen_task', 'denoise',           # VG-OCR-Denoise: OCR nhiễu + thị giác → sinh bản SẠCH (sửa lỗi)
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN gen_all
Vision unfreeze đang **TẮT** (`--vision_unfreeze_last_n 0`) để có bản pretrain **ổn định** (validate gen_all + whole-word trước). Theo dõi **`loss_mlm` và `loss_gen` GIẢM DẦN** trong log eval — đó là tín hiệu decoder đang học. Khi ổn định sẽ bật lại vision unfreeze kèm biện pháp ổn định (LR nhỏ cho vision / gradient checkpointing).

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm (vision instability). Bật lại (2-4) SAU khi có bản ổn định + stabilize
    '--num_train_epochs', '3',
])

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/pretrain', repo_id=HF_PRETRAIN_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded pretrain ->', HF_PRETRAIN_REPO)